In [2]:
import pandas as pd
import numpy as np
import sqlite3
import os

In [4]:
df = pd.read_csv('../data/raw/retail_store_sales.csv')
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [5]:
print('Shape:', df.shape)
print('\nColumn Names:\n', df.columns.tolist())
print('\nData Types:\n', df.dtypes)

Shape: (12575, 11)

Column Names:
 ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied']

Data Types:
 Transaction ID       object
Customer ID          object
Category             object
Item                 object
Price Per Unit      float64
Quantity            float64
Total Spent         float64
Payment Method       object
Location             object
Transaction Date     object
Discount Applied     object
dtype: object


# Full Diagnostic

In [6]:
print('=== MISSING VALUES ===')
print(df.isnull().sum())

print('\n=== DUPLICATE ROWS ===')
print(df.duplicated().sum())

print('\n=== UNIQUE VALUES ===')
for col in ['Category', 'Item', 'Payment Method', 'Location']:
    if col in df.columns:
        print(f'\n{col}: {df[col].unique()}')

=== MISSING VALUES ===
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

=== DUPLICATE ROWS ===
0

=== UNIQUE VALUES ===

Category: ['Patisserie' 'Milk Products' 'Butchers' 'Beverages' 'Food' 'Furniture'
 'Electric household essentials' 'Computers and electric accessories']

Item: ['Item_10_PAT' 'Item_17_MILK' 'Item_12_BUT' 'Item_16_BEV' 'Item_6_FOOD'
 nan 'Item_1_FOOD' 'Item_16_FUR' 'Item_22_BUT' 'Item_3_BUT' 'Item_2_FOOD'
 'Item_24_PAT' 'Item_16_MILK' 'Item_17_PAT' 'Item_13_EHE' 'Item_7_BEV'
 'Item_4_EHE' 'Item_10_FOOD' 'Item_14_FUR' 'Item_20_BUT' 'Item_25_FUR'
 'Item_14_FOOD' 'Item_22_PAT' 'Item_11_FOOD' 'Item_6_PAT' 'Item_21_EHE'
 'Item_25_BEV' 'Item_23_FOOD' 'Item_10_FUR' 'Item_11_BEV' 'Item_23_BUT'
 'Item_22_BEV' 'Item_10_EHE' 'Item_24_BUT' 'Ite

# Cleaning Pipeline

In [8]:
both_missing = df['Quantity'].isna() & df['Total Spent'].isna()
only_quantity = df['Quantity'].isna() & df['Total Spent'].notna()
only_total = df['Quantity'].notna() & df['Total Spent'].isna()

print("Both missing: ", both_missing.sum())
print('Only Quantity missing: ', only_quantity.sum())
print('Only Total Spent missing', only_total.sum())

Both missing:  604
Only Quantity missing:  0
Only Total Spent missing 0


In [9]:
df_clean = df.copy()

In [16]:
string_cols = ['Transaction ID', 'Customer ID', 'Category', 'Item',
    'Payment Method', 'Location']
df_clean[string_cols] = df_clean[string_cols].apply(
    lambda x: x.str.strip() if x.dtype == "object" else x
)

df_clean['Transaction Data'] = pd.to_datetime(
    df_clean['Transaction Date'], errors='coerce'
)

df_clean['Discount Applied'] = df_clean['Discount Applied'].fillna(False).astype(bool)

df_clean['Item'] = df_clean['Item'].fillna('Unknown Item')

reconstruct_qty = (
    only_quantity &
    df_clean['Price Per Unit'].notna() 
)
df_clean.loc[reconstruct_qty, 'Quantity'] = np.round(
    df_clean['Total Spent'] / df_clean["Price Per Unit"]
)

reconstruct_total = (
    only_total &
    df_clean['Price Per Unit'].notna()  
)
df_clean.loc[reconstruct_total, 'Total Spent'] = np.round(
    df_clean['Price Per Unit'] / df_clean["Quantity"]
)

df_clean['quantity_imputed'] = reconstruct_qty
df_clean['total_imputed'] = reconstruct_total

unrecoverable = df_clean[df_clean['Total Spent'].isna() |
                         df_clean['Quantity'].isna()].copy()

df_clean = df_clean[
    df_clean['Total Spent'].notna() & df_clean['Quantity'].notna()
].copy()

In [17]:
df_clean.isnull().sum()

Transaction ID           0
Customer ID              0
Category                 0
Item                     0
Price Per Unit         609
Quantity                 0
Total Spent              0
Payment Method           0
Location                 0
Transaction Date         0
Discount Applied         0
Transaction Data         0
quantity_imputed         0
total_imputed            0
total_spent_imputed      0
dtype: int64

In [18]:
df_clean['formula_check'] = abs(
    (df_clean['Price Per Unit'] * df_clean['Quantity']) - df_clean['Total Spent']
)

violations = df_clean[df_clean['formula_check'] > 0.01]
clean_rows = df_clean[df_clean['formula_check'] <= 0.01]

print(f"Formula holds: {len(clean_rows)} rows")
print(f"Formula violations: {len(violations)} rows")

if len(violations) > 0:
    print("\nSample violations:")
    print(violations[['Transaction ID', 'Price Per Unit', 
                       'Quantity', 'Total Spent', 'formula_check']].head(10))

Formula holds: 11362 rows
Formula violations: 0 rows


In [19]:
df_clean.to_csv("../data/clean/nzimande_sales_clean.csv", index=False)

unrecoverable.to_csv("../data/clean/nzimande_unrecoverable_rows.csv", index=False)

# SQLite Database Ingestion

In [21]:
db_path = '../database/nzimande_wholesales.db'

conn = sqlite3.connect(db_path)

df_clean.to_sql(
    'sales', 
    conn, 
    if_exists = "replace", 
    index=False
)

conn.close()

In [22]:
print("Rows in df_clean:", len(df_clean))
print("Rows in saved CSV:", len(pd.read_csv("../data/clean/nzimande_sales_clean.csv")))

Rows in df_clean: 11971
Rows in saved CSV: 11971


In [24]:
# Remove rows where Price Per Unit is still null
# These rows cannot be fully audited (formula cannot be verified)
df_clean = df_clean[df_clean['Price Per Unit'].notna()].copy()

print(f"Final clean rows: {len(df_clean)}")

# Verify formula holds on the true final dataset
df_clean['formula_check'] = abs(
    (df_clean['Price Per Unit'] * df_clean['Quantity']) - df_clean['Total Spent']
)
violations = df_clean[df_clean['formula_check'] > 0.01]
print(f"Formula violations: {len(violations)}")

# Overwrite the clean CSV
df_clean.to_csv("../data/clean/nzimande_sales_clean.csv", index=False)
print("Clean CSV overwritten.")

# Re-ingest into SQLite
conn = sqlite3.connect("../database/nzimande_wholesales.db")
df_clean.to_sql(
    "sales",
    conn,
    if_exists="replace",
    index=False
)
conn.close()
print("Database re-ingested with corrected dataset.")

Final clean rows: 11362
Formula violations: 0
Clean CSV overwritten.
Database re-ingested with corrected dataset.


# Tableau Dashboard

In [25]:
df_clean.to_csv("../exports/nzimande_tableau.csv", index=False)
print("Tableau-ready export created.")

Tableau-ready export created.
